In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon
import numpy as np
from pathlib import Path
import os
import sys

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)

BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


#### read the cleaned dataset containing OHCA incidences with their respective latitude and longtitude

In [ ]:
dataset_filepath = str(BASE_DATASET_PATH / "3_OHCA_with_coords.xlsx")
ohca_df = pd.read_excel(dataset_filepath)
print(ohca_df.shape)

In [ ]:
columns_to_drop = ["Unnamed: 0.2", "Unnamed: 0.1", "Unnamed: 0", "Location Type Other_x", "Location Type Other_y"]
ohca_df = ohca_df.drop(columns = columns_to_drop)
ohca_df.head(3)

In [ ]:
ohca_geo_df = gpd.GeoDataFrame(
    ohca_df,
    geometry = gpd.points_from_xy(
        ohca_df["lon"],
        ohca_df["lat"]
    ),
    crs = "EPSG:4326"
)

# reproject to 3414, use by Singapore
ohca_geo_df = ohca_geo_df.to_crs(3414)
ohca_geo_df.head(3)

# make a geometry column 
# gdf_scene_coordinates = gpd.GeoDataFrame(
#     df_scene_coordinates,
#     geometry = gpd.points_from_xy(
#         df_scene_coordinates["combined scene longitude"],
#         df_scene_coordinates["combined scene latitude"]        
#     ),
#     crs="EPSG:4326"  # WGS84 lat/lon
# )
# # project to 3375, used for malaysia (units: metres)
# gdf_scene_coordinates = gdf_scene_coordinates.to_crs("EPSG:3375")

In [ ]:
# save to folder
output_filepath = str(BASE_DATASET_PATH / "3_OHCA_with_geometric_pts.csv")
ohca_geo_df.to_csv(output_filepath)

gpkg_output_filepath = str(BASE_DATASET_PATH / "3_OHCA_with_geometric_pts.gpkg")
ohca_geo_df.to_file(gpkg_output_filepath)

#### Adding 3_OHCA_with_geometric_pts.gpkg to sql for intersection with hectare grids

In [ ]:
def construct_add_command(filename: str, database_name: str, username: str):

    cmd = [f"ogr2ogr -f PostgreSQL PG:\"dbname={database_name} user={username}\" ", 
           filename + ".gpkg",
           " -nln ", filename,
           " -lco GEOMETRY_NAME=geom ",
           "-lco FID=id ",
           "-nlt PROMOTE_TO_MULTI ",
           filename
    ]
    result = "".join(cmd)
    return result


def add_to_postgres(database_name: str, username: str, filename: str | None = None):
    """
    adds the layers from .gpkg file to postgreSQL
    Can't get PostgreSQL command to work,
    need to manually copy command output and run in terminal
    """

    cmd = construct_add_command(filename, database_name, username)
    print(cmd)

    cmd_indexes = f"""
CREATE INDEX IF NOT EXISTS hectare_grid_geom_idx ON hectare_grid USING GIST (geom);

CREATE INDEX IF NOT EXISTS roads_polygon_geom_idx ON {filename} USING GIST (geom);
    """
    print(cmd_indexes)


add_to_postgres("postgis", "yitong", "3_OHCA_with_geometric_pts")

#### intersect with hectare table

In [ ]:
def intersect_ohca_with_hectares(output_table_name: str, input_table_name: str):
    cmd = f"""
DROP TABLE IF EXISTS {output_table_name};

CREATE TABLE {output_table_name} AS
SELECT 
    g.id AS grid_id,
    g.row_index AS grid_row,
    g.col_index AS grid_col,
    mp.id AS id,
    mp."case _" AS case_num,
    mp."date of incident" AS datetime,
    mp."location type" AS location_type,
    mp."location type other" AS location_type_other,
    mp."time call received at dispatch center" AS time,
    mp."lat" as lat,
    mp."lon" as lon,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN {'"'+input_table_name+'"'} mp
  ON ST_Intersects(g.geom, mp.geom);
"""

    print(cmd)

intersect_ohca_with_hectares("ohca_by_hectare", "3_ohca_with_geometric_pts")

#### Added a date column to the ohca_by_hectare table
ALTER TABLE ohca_by_hectare ADD date DATE;

#### Populate `date` column with the dates from the `datetime` column
UPDATE ohca_by_hectare SET date = DATE(datetime);

In [ ]:
def extract_table_into_file(database_name: str, username: str, table_name: str):
    cmd = f"""
ogr2ogr -f GPKG {table_name}.gpkg PG:"dbname={database_name} user={username}" {table_name}
    """
    print(cmd)

In [ ]:
extract_table_into_file("postgis", "yitong", "ohca_by_hectare")

### The codes above only need to be ran once to obtain the ohca_by_hectare table. The codes below will be ran multiple times for each year. 
#### convert the grid_classification file obtained from the using_openstreetmaps.ipynb notebook into csv so that it can be added to postgres

In [30]:
def convert_excel_to_csv(start_year: int, end_year: int, excel_file: str):
    year = start_year

    for i in range(end_year - start_year + 1):
        excel_filepath = Path(BASE_DATASET_PATH / f"geospatial_data/{year + i}_geospatial/{excel_file}.xlsx")
        df = pd.read_excel(excel_filepath)
        csv_filepath = Path(BASE_DATASET_PATH / f"geospatial_data/{year + i}_geospatial/{excel_file}.csv")
        df.to_csv(csv_filepath)
        # print(excel_filepath)
        # print(csv_filepath)
    print("Done")


In [ ]:
convert_excel_to_csv(2014, 2020, "grid_classification")

Done


#### Adding the csv file to postgres

In [22]:
def add_to_postgres(filename: str, year: str, dbname: str, username: str):
    cmd = f"""
    ogr2ogr -f "PostgreSQL" PG:"dbname={dbname} user={username}" {filename}.csv -nln {filename}_{year}
    """
    print(cmd)

add_to_postgres("grid_classification", "2019", "postgis", "yitong")


    ogr2ogr -f "PostgreSQL" PG:"dbname=postgis user=yitong" grid_classification.csv -nln grid_classification_2019
    


Typecast the columns so that they can be compared with the ohca_by_hectare table. Else the columns will be varchar

In [16]:
def typecast_columns(table_name: str):
    cmd = f"""
ALTER TABLE {table_name} 
ALTER COLUMN grid_id TYPE integer USING grid_id::integer,
ALTER COLUMN grid_row TYPE bigint USING grid_row::bigint,
ALTER COLUMN grid_col TYPE bigint USING grid_col::bigint;
"""
    print(cmd)

typecast_columns("grid_classification_2015")


ALTER TABLE grid_classification_2015 
ALTER COLUMN grid_id TYPE integer USING grid_id::integer,
ALTER COLUMN grid_row TYPE bigint USING grid_row::bigint,
ALTER COLUMN grid_col TYPE bigint USING grid_col::bigint;



#### combine ohca incidence with classification of hectare grid by year


In [36]:
def combine_ohca_with_hectare_grid(year: str):
    cmd = f"""
DROP TABLE IF EXISTS ohca_by_hectare_{year};

CREATE TABLE ohca_by_hectare_{year} AS
SELECT
    g.grid_id AS grid_id,
    g.grid_row AS grid_row,
    g.grid_col AS grid_col,
    g.id AS id,
    g.date AS date,
    g.time AS time,
    g.location_type AS location_type,
    g.location_type_other AS location_type_other,
    c.hectare_classification AS hectare_classification
FROM
    ohca_by_hectare g
JOIN
    grid_classification_{year} c
ON
    g.grid_id = c.grid_id
WHERE
    EXTRACT(YEAR FROM g.date) = {year};
"""
    print(cmd)

In [41]:
combine_ohca_with_hectare_grid("2019")


DROP TABLE IF EXISTS ohca_by_hectare_2019;

CREATE TABLE ohca_by_hectare_2019 AS
SELECT
    g.grid_id AS grid_id,
    g.grid_row AS grid_row,
    g.grid_col AS grid_col,
    g.id AS id,
    g.date AS date,
    g.time AS time,
    g.location_type AS location_type,
    g.location_type_other AS location_type_other,
    c.hectare_classification AS hectare_classification
FROM
    ohca_by_hectare g
JOIN
    grid_classification_2019 c
ON
    g.grid_id = c.grid_id
WHERE
    EXTRACT(YEAR FROM g.date) = 2019;



#### Export out the table as geopackage

In [42]:
def export_to_gpkg(year: str):
    cmd = f"""
ogr2ogr -f CSV ohca_by_hectare_{year}.csv PG:"dbname=postgis user=yitong" ohca_by_hectare_{year}
"""
    print(cmd)

In [49]:
export_to_gpkg("2015")


ogr2ogr -f CSV ohca_by_hectare_2015.csv PG:"dbname=postgis user=yitong" ohca_by_hectare_2015

